In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Load Dataset

In [2]:
import random
from torch.utils.data import Subset

data_dir = "./imagenet_validation"      # Download from https://www.kaggle.com/datasets/tusonggao/imagenet-validation-dataset

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

n_samples = 5000
indices = random.sample(range(len(dataset)), n_samples)
subset = Subset(dataset, indices)

loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

num_classes = len(dataset.classes)
print("Dataset size:", len(loader.dataset))
print("Classes:", num_classes)

Dataset size: 5000
Classes: 1000


# Evaluate Models

In [7]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoProcessor


class ModelWrapper(nn.Module):
    def __init__(self, model_name, num_classes, device):
        super().__init__()
        self.name = model_name
        self.device = device

        # ===== ViT (timm supervised) =====
        if model_name == "vit_base":
            self.model = timm.create_model(
                "vit_base_patch16_224", pretrained=True, num_classes=num_classes
            )
        elif model_name == "vit_large":
            self.model = timm.create_model(
                "vit_large_patch16_224", pretrained=True, num_classes=num_classes
            )

        # ===== DINOv2 =====
        elif model_name == "dinov2_base":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_lc")
        elif model_name == "dinov2_large":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14_lc")

        else:
            raise ValueError(f"Unknown model: {model_name}")

        self.model = self.model.to(device)
        self.eval()

    def forward(self, images):
        with torch.no_grad():
            return self.model(images)

In [ ]:
from pathlib import Path
import torchvision.transforms.functional as TF
import time


def apply_affine_batch(images, angle, translate, scale, shear):
    return torch.stack(
        [
            TF.affine(img, angle=angle, translate=translate, scale=scale, shear=shear)
            for img in images
        ]
    )


def evaluate(
    model,
    loader,
    model_name: str = "Model",
    translation: tuple[float, float] = [0, 0],
    rotation: float = 0.0,
    scale: float = 1.0,
    shear: tuple[float, float] = [0, 0],
    save_results_path: str = "results.csv",
):
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = apply_affine_batch(
                images, angle=rotation, translate=translation, scale=scale, shear=shear
            )

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds_top1 = outputs.argmax(dim=1)
            correct_top1 += (preds_top1 == labels).sum().item()

            _, preds_top5 = outputs.topk(5, dim=1)
            correct_top5 += (preds_top5 == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    end_time = time.time()
    eval_time = end_time - start_time

    acc_top1 = correct_top1 / total
    acc_top5 = correct_top5 / total

    results_path = Path(save_results_path)
    header = (
        "Model Name,Dataset Size,Translation X, Translation Y,"
        "Rotation,Scale,Shear X, Shear Y,Top-1 Accuracy,Top-5 Accuracy,Time"
    )

    if not results_path.exists() or results_path.stat().st_size == 0:
        # file doesn't exist or is empty -> create with header
        with results_path.open("w", encoding="utf-8") as f:
            f.write(header + "\n")
    else:
        # file exists, check header consistency
        with results_path.open("r", encoding="utf-8") as f:
            first_line = f.readline().strip()
        if first_line != header:
            with results_path.open("w", encoding="utf-8") as f:
                f.write(header + "\n")

    # append result row
    with results_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{model_name},{len(loader.dataset)},"
            f"{translation[0]},{translation[1]},{rotation},{scale},"
            f"{shear[0]},{shear[1]},{acc_top1:.4f},{acc_top5:.4f},{eval_time:.2f}\n"
        )

    return acc_top1, acc_top5

In [9]:
def evaluate_architecture(model_name, loader, geometric_transforms):
    model = ModelWrapper(model_name, num_classes, device)

    # Baseline evaluation without transformations
    if geometric_transforms["baseline"]:
        print(f"Baseline (No Transformations):")
        acc_top1, acc_top5 = evaluate(model, loader, model_name)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Translations
    for translation in geometric_transforms["translations"]:
        print(f"Translation {translation}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, translation=translation)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Rotations
    for rotation in geometric_transforms["rotations"]:
        print(f"Rotation {rotation}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, rotation=rotation)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Scales
    for scale in geometric_transforms["scales"]:
        print(f"Scale {scale}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, scale=scale)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Shears
    for shear in geometric_transforms["shears"]:
        print(f"Shear {shear}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, shear=shear)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

### ViT Base

In [10]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_base", loader, geometric_transforms)

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:33<00:00,  2.39it/s]


Top-1 Accuracy: 81.62%
Top-5 Accuracy: 95.30%
Translation (25, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:18<00:00,  4.35it/s]


Top-1 Accuracy: 81.10%
Top-5 Accuracy: 95.66%
Translation (0, 25):


...

Top-1 Accuracy: 58.46%
Top-5 Accuracy: 78.96%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:18<00:00,  4.33it/s]


Top-1 Accuracy: 57.76%
Top-5 Accuracy: 77.64%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:19<00:00,  3.97it/s]

Top-1 Accuracy: 58.62%
Top-5 Accuracy: 79.52%


### ViT Large

In [12]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("vit_large", loader, geometric_transforms)

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:46<00:00,  1.72it/s]


Top-1 Accuracy: 84.68%
Top-5 Accuracy: 96.98%
Translation (25, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:46<00:00,  1.70it/s]


...

Top-1 Accuracy: 68.50%
Top-5 Accuracy: 87.28%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:46<00:00,  1.70it/s]


Top-1 Accuracy: 69.28%
Top-5 Accuracy: 87.30%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:47<00:00,  1.67it/s]

Top-1 Accuracy: 68.76%
Top-5 Accuracy: 86.80%


### DINOv2 Base

In [13]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("dinov2_base", loader, geometric_transforms)

Using cache found in /zhome/c7/a/228503/.cache/torch/hub/facebookresearch_dinov2_main
/zhome/c7/a/228503/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/zhome/c7/a/228503/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/zhome/c7/a/228503/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:22<00:00,  3.47it/s]


Top-1 Accuracy: 84.80%
Top-5 Accuracy: 96.74%
Translation (25, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:22<00:00,  3.56it/s]


...

Top-1 Accuracy: 60.28%
Top-5 Accuracy: 81.60%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:22<00:00,  3.45it/s]


Top-1 Accuracy: 58.96%
Top-5 Accuracy: 80.98%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:23<00:00,  3.31it/s]

Top-1 Accuracy: 58.86%
Top-5 Accuracy: 81.28%


### DINOv2 Large

In [14]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("dinov2_large", loader, geometric_transforms)

Using cache found in /zhome/c7/a/228503/.cache/torch/hub/facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /zhome/c7/a/228503/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████████████████████████████████████████████████████████████████████████| 1.13G/1.13G [00:20<00:00, 60.6MB/s]


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_linear4_head.pth" to /zhome/c7/a/228503/.cache/torch/hub/checkpoints/dinov2_vitl14_linear4_head.pth


100%|██████████████████████████████████████████████████████████████████████████████| 19.5M/19.5M [00:00<00:00, 56.3MB/s]


...

Top-1 Accuracy: 73.10%
Top-5 Accuracy: 90.72%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:59<00:00,  1.32it/s]


Top-1 Accuracy: 71.42%
Top-5 Accuracy: 90.30%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.29it/s]

Top-1 Accuracy: 72.70%
Top-5 Accuracy: 90.60%
